In [1]:
import os

# Your large volume is mounted at /workspace.
# We'll create a dedicated folder inside it for the cache.
cache_dir = "/workspace/huggingface_cache"
os.environ['HF_HOME'] = cache_dir

# Create the directory if it doesn't exist to be safe
os.makedirs(cache_dir, exist_ok=True)

print(f"✅ Hugging Face cache directory is now set to: {os.environ['HF_HOME']}")

✅ Hugging Face cache directory is now set to: /workspace/huggingface_cache


In [2]:
# Step 1: Install all required Python packages
!pip install huggingface_hub transformers accelerate einops hf_transfer # <-- Added hf_transfer here

# Step 2: Clone the repository if it doesn't exist, or pull the latest changes if it does.
!if [ -d "repository/circuit-tracer" ]; then \
    echo "✅ Repository found. Pulling latest changes..."; \
    (cd repository/circuit-tracer && git pull); \
else \
    echo "Cloning repository for the first time..."; \
    mkdir -p repository && git clone https://github.com/safety-research/circuit-tracer repository/circuit-tracer; \
fi

# Step 3: Install the package from the local repository
!pip install ./repository/circuit-tracer

  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached accelerate-1.11.0-py3-none-any.whl.metadata (19 kB)
  Using cached einops-0.8.1-py3-none-any.whl.metadata (13 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached hf_xet-1.1.10-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.7 kB)
  Using cached regex-2025.10.23-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
  Using cached safetensors-0.6.2-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
Using cached hf_xet-1.1.10-cp37-abi3-manylinux_2_

In [3]:
import os
import sys
import subprocess
from huggingface_hub import login

# Add the cloned repository to the Python path
sys.path.append('repository/circuit-tracer')
sys.path.append('repository/circuit-tracer/demos')

# --- OPTIMIZED LOGIN LOGIC ---
try:
    # Check if we're already logged in by running the 'whoami' command.
    # This avoids a new login API call if we don't need one.
    subprocess.run(["huggingface-cli", "whoami"], check=True, capture_output=True, text=True)
    print("✅ Already logged into Hugging Face.")

except (subprocess.CalledProcessError, FileNotFoundError):
    # This block runs if the 'whoami' command fails, meaning we are not logged in.
    print("Not logged in. Attempting to log in now...")
    
    # Get the HF_TOKEN from the environment variable
    hf_token = os.getenv('HF_TOKEN')

    if hf_token:
        login(token=hf_token)
        print("✅ Successfully logged into Hugging Face.")
    else:
        print("⚠️ HF_TOKEN not found. Cannot log in.")

✅ Already logged into Hugging Face.


In [4]:
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [5]:
model_name = 'google/gemma-2-2b'
transcoder_name = "gemma"
model = ReplacementModel.from_pretrained(model_name, transcoder_name, dtype=torch.bfloat16)

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


In [6]:
import transformer_lens
from pathlib import Path
import os
import torch

# --- 1. Mitigate Memory Fragmentation ---
# This can help PyTorch manage memory more efficiently.
os.environ['PYTOOLKIT_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# --- 2. Configuration ---
initial_prompt = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Let's break this down. First, we need to consider the initial conditions and then"
num_steps_to_generate = 5 # Increased slightly to see more of the CoT

# Attribution parameters
max_n_logits = 10
desired_logit_prob = 0.95
max_feature_nodes = 8192
batch_size = 64 # Kept the smaller batch size to avoid memory issues
offload = 'disk'
verbose = False

# Node and edge thresholds for the visualization files
node_threshold = 0.8
edge_threshold = 0.98

# --- 3. Setup Output Directories ---
graph_pt_dir = Path('graphs_iterative').resolve()
graph_files_dir = Path('graph_files_iterative').resolve()
graph_pt_dir.mkdir(exist_ok=True)
graph_files_dir.mkdir(exist_ok=True)

# --- 4. Iteration Loop ---
current_prompt = initial_prompt
# We need direct access to the tokenizer for decoding, which is on the base model
tokenizer = model.tokenizer

print(f"Starting iterative generation from prompt: '{current_prompt}'")
print("-" * 30)

for i in range(num_steps_to_generate):
    print(f"▶️ Step {i+1}/{num_steps_to_generate}")
    print(f"   Input Prompt: '{current_prompt}'")

    # Generate the attribution graph for the model's *next* predicted token
    graph = attribute(
        prompt=current_prompt,
        model=model,
        max_n_logits=max_n_logits,
        desired_logit_prob=desired_logit_prob,
        batch_size=batch_size,
        max_feature_nodes=max_feature_nodes,
        offload=offload,
        verbose=verbose
    )

    # --- NEW: Extract the next token directly from the graph ---
    # The graph stores the logits for the final token position.
    # We can find the token with the highest probability (the one the model would choose).
    #print(graph.input_tokens,graph.logit_tokens, graph.logit_probabilities )
    
    predicted_token_strings = [tokenizer.decode([token_id]) for token_id in graph.logit_tokens]
    print(predicted_token_strings)
    new_token_str = predicted_token_strings[0]
    

    print(f"   Generated Token: '{new_token_str}'")

    # Create a unique name (slug) for the graph files
    safe_token_str = new_token_str.strip().replace(" ", "_").replace("\n", "_").replace("<0x0A>", "NL")
    slug = f"step_{i:02d}_{safe_token_str}"
    graph_pt_path = graph_pt_dir / f"{slug}.pt"
    
    # Save the graph object
    print(f"   Saving graph to: {graph_pt_path}")
    graph.to_pt(graph_pt_path)

    # Create the visualization files
    create_graph_files(
        graph_or_path=graph_pt_path,
        slug=slug,
        output_path=graph_files_dir,
        node_threshold=node_threshold,
        edge_threshold=edge_threshold
    )

    # Update the prompt for the next iteration
    current_prompt += predicted_token_strings[0]
    print("-" * 30)

print("✅ Finished generating all graphs.")

Starting iterative generation from prompt: 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Let's break this down. First, we need to consider the initial conditions and then'
------------------------------
▶️ Step 1/5
   Input Prompt: 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Let's break this down. First, we need to consider the initial conditions and then'


sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


[' we', ' the', ' solve', ' use', ' work', ' apply', ' determine', ' set', ',', ' make']
   Generated Token: ' we'
   Saving graph to: /workspace/graphs_iterative/step_00_we.pt
------------------------------
▶️ Step 2/5
   Input Prompt: 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Let's break this down. First, we need to consider the initial conditions and then we'
[' need', ' can', "'", ' will', ' have', ' are', ' want', ' apply', ' should', ' use']
   Generated Token: ' need'
   Saving graph to: /workspace/graphs_iterative/step_01_need.pt
------------------------------
▶️ Step 3/5
   Input Prompt: 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Let's break this down. First, we need to consider the initial conditions and then we need'
[' to']
   Generated Token: ' to'
   Saving graph to: /workspace/graphs_iterative/step_02_to.pt
------------------------------
▶️ S